# Chemical Space Analysis with t-SNE

This notebook generates t-SNE (t-Distributed Stochastic Neighbor Embedding) visualizations for chemical space analysis of the DeNovo PfHDAC1 HIGHDIV dataset.

## Notebook Structure:

1. **Imports** - Required libraries (sklearn, rdkit, bokeh)
2. **Data Loading** - Reading the highdiv dataset
3. **Fingerprint Functions** - MACCS and ECFP4
4. **create_tsne_data Function** - Calculates t-SNE for a dataset
5. **Individual t-SNE Calculation** - Processes each sub-dataset separately
6. **Individual Interactive Plots** - Generates HTML visualizations with Bokeh
7. **Combined Visualization** - Concatenates all sub-datasets, removes duplicates by InChI, and generates visualizations with legend by source dataset

## Processed Datasets:
- **Descriptors**: MACCS, ECFP4
- **Sub-datasets**: CONCAT, CRAFT, ENAMINE, LANAPDB, MAYBRIDGE

**Total**: 12 visualizations (HTML)
- 10 individual (5 sub-datasets × 2 descriptors)
- 2 combined

In [1]:
from sklearn.manifold import TSNE
import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys
from rdkit.Chem import Draw
from bokeh.plotting import figure, save, output_file
from bokeh.models import HoverTool, ColumnDataSource
import base64
from io import BytesIO

In [2]:
df_highdiv = pd.read_csv('/Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/Analysis/concat_datasets_highdiv.csv')

In [3]:
def smiles_to_maccs(smiles):
    """Converts a SMILES into MACCS fingerprint"""
    mol = Chem.MolFromSmiles(smiles)
    return MACCSkeys.GenMACCSKeys(mol) if mol else None

def smiles_to_ECFP4(smiles):
    """Converts a SMILES into ECFP4 fingerprint (Morgan with radius 2)"""
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024) if mol else None

In [4]:
def create_tsne_data(df, fingerprint_type='maccs', perplexity=30, n_iter=1000):
    """
    Creates data needed to generate a t-SNE plot
    
    Args:
        df: DataFrame with 'SMILES' column
        fingerprint_type: 'maccs' or 'ecfp4'
        perplexity: The perplexity is related to the number of nearest neighbors that is used in other manifold learning algorithms.
        n_iter: Maximum number of iterations for the optimization. Should be at least 250.
    
    Returns:
        layout: t-SNE embedded coordinates (n_samples, 2)
        df: Updated DataFrame with fingerprints
    """
    # Generate fingerprints
    if fingerprint_type == 'maccs':
        df['fingerprints'] = [smiles_to_maccs(smiles) for smiles in df['SMILES']]
    elif fingerprint_type == 'ecfp4':
        df['fingerprints'] = [smiles_to_ECFP4(smiles) for smiles in df['SMILES']]
    else:
        raise ValueError("fingerprint_type must be 'maccs' or 'ecfp4'")
    
    # Remove rows without fingerprints
    df = df.dropna(subset=['fingerprints'])
    
    # Convert fingerprints to numpy array for sklearn
    # Each fingerprint is converted to a list of bits (0s and 1s)
    X = np.array([list(fp) for fp in df['fingerprints']])
    
    # Calculate t-SNE
    # Using 'jaccard' metric would be ideal for binary fingerprints, but it can be slow (O(N^2)).
    # For speed 'euclidean' is often used as a proxy, or precomputed distance matrix.
    # Given dataset sizes are small (<few k), we can try using a metric suitable for binary data if needed,
    # but default initialization usually works well.
    tsne = TSNE(n_components=2, verbose=1, perplexity=perplexity, n_iter=n_iter, random_state=42, init='pca', learning_rate='auto')
    layout = tsne.fit_transform(X)
    
    return layout, df

In [5]:
# Create results directory if it doesn't exist
import os
output_dir = '/Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/Analysis/Results/Chemical_Spaces/TSNE'
os.makedirs(output_dir, exist_ok=True)

# Dictionary to store calculated t-SNE results
tsne_results = {}

fingerprint_types = ['MACCS', 'ECFP4']
dataset_names = ['CONCAT', 'CRAFT', 'ENAMINE', 'LANAPDB', 'MAYBRIDGE']

print("="*60)
print("CALCULATING t-SNEs")
print("="*60)
print(f"\nProcessing: HIGHDIV ({len(df_highdiv)} molecules total)")
print('='*60)

for dataset_name in dataset_names:
    # Filter specific sub-dataset
    df_subset = df_highdiv[df_highdiv['Dataset'].str.contains(dataset_name, case=False, na=False)].copy()

    if len(df_subset) == 0:
        print(f"⚠️  {dataset_name}: No data found")
        continue

    print(f"\n{dataset_name}: {len(df_subset)} molecules")

    for descriptor in fingerprint_types:
        print(f"  - Calculating t-SNE with {descriptor}...", end=' ')

        try:
            # Adjust perplexity based on dataset size if needed
            n_samples = len(df_subset)
            perp = min(30, max(5, n_samples // 10))  # Heuristic adjustment

            # Create t-SNE
            layout, df_processed = create_tsne_data(df_subset, fingerprint_type=descriptor.lower(), perplexity=perp)

            # Store results
            key = f"HIGHDIV_{descriptor}_{dataset_name}"
            tsne_results[key] = {
                'layout': layout,
                'df': df_processed,
                'descriptor': descriptor,
                'dataset': dataset_name,
                'perplexity': perp
            }

            print(f"✓ Complete ({len(df_processed)} molecules)")

        except Exception as e:
            print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Calculation complete! {len(tsne_results)} t-SNEs calculated.")
print('='*60)

CALCULATING t-SNEs

Processing: HIGHDIV (2712 molecules total)

CONCAT: 1288 molecules
  - Calculating t-SNE with MACCS... 

/opt/anaconda3/envs/ia/lib/python3.13/site-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 1288 samples in 0.000s...
[t-SNE] Computed neighbors for 1288 samples in 0.205s...
[t-SNE] Computed conditional probabilities for sample 1000 / 1288
[t-SNE] Computed conditional probabilities for sample 1288 / 1288
[t-SNE] Mean sigma: 1.742649
[t-SNE] KL divergence after 250 iterations with early exaggeration: 69.180168
[t-SNE] KL divergence after 1000 iterations: 0.871098
✓ Complete (1288 molecules)
  - Calculating t-SNE with ECFP4... 

[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerator
[16:17:12] DEPRECATION WARNING: please use MorganGenerat

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 1288 samples in 0.000s...
[t-SNE] Computed neighbors for 1288 samples in 0.127s...
[t-SNE] Computed conditional probabilities for sample 1000 / 1288
[t-SNE] Computed conditional probabilities for sample 1288 / 1288
[t-SNE] Mean sigma: 2.295225
[t-SNE] KL divergence after 250 iterations with early exaggeration: 69.463188
[t-SNE] KL divergence after 1000 iterations: 0.790197
✓ Complete (1288 molecules)

CRAFT: 394 molecules
  - Calculating t-SNE with MACCS... 

...

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 356 samples in 0.000s...
[t-SNE] Computed neighbors for 356 samples in 0.011s...
[t-SNE] Computed conditional probabilities for sample 356 / 356
[t-SNE] Mean sigma: 2.053955
[t-SNE] KL divergence after 250 iterations with early exaggeration: 64.747780
[t-SNE] KL divergence after 1000 iterations: 0.634198
✓ Complete (356 molecules)

MAYBRIDGE: 323 molecules
  - Calculating t-SNE with MACCS... 

/opt/anaconda3/envs/ia/lib/python3.13/site-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 323 samples in 0.000s...
[t-SNE] Computed neighbors for 323 samples in 0.005s...
[t-SNE] Computed conditional probabilities for sample 323 / 323
[t-SNE] Mean sigma: 2.067120
[t-SNE] KL divergence after 250 iterations with early exaggeration: 55.725357
[t-SNE] KL divergence after 1000 iterations: 0.536784
✓ Complete (323 molecules)
  - Calculating t-SNE with ECFP4... 

[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerator
[16:17:21] DEPRECATION WARNING: please use MorganGenerat

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 323 samples in 0.000s...
[t-SNE] Computed neighbors for 323 samples in 0.010s...
[t-SNE] Computed conditional probabilities for sample 323 / 323
[t-SNE] Mean sigma: 2.258364
[t-SNE] KL divergence after 250 iterations with early exaggeration: 61.029285
[t-SNE] KL divergence after 1000 iterations: 0.591176
✓ Complete (323 molecules)

Calculation complete! 10 t-SNEs calculated.


In [6]:
# Function to convert molecule to base64 image
def mol_to_base64(smiles, size=(150, 150)):
    """Converts SMILES to base64 encoded PNG image"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return ""
        img = Draw.MolToImage(mol, size=size)
        buffered = BytesIO()
        img.save(buffered, format="PNG")
        img_str = base64.b64encode(buffered.getvalue()).decode()
        return f"data:image/png;base64,{img_str}"
    except:
        return ""

# Generate interactive plots with Bokeh
print("="*60)
print("GENERATING INTERACTIVE PLOTS (HTML)")
print("="*60)

for key, data in tsne_results.items():
    layout = data['layout']
    df_processed = data['df']
    descriptor = data['descriptor']
    dataset_name = data['dataset']
    perplexity = data.get('perplexity', 'N/A')

    print(f"Plotting: {key}...", end=' ')

    try:
        # Extract coordinates from t-SNE layout (n_samples, 2)
        x = layout[:, 0]
        y = layout[:, 1]

        # Prepare data for Bokeh
        df_plot = df_processed.copy().reset_index(drop=True)

        # Remove fingerprints column which is not serializable
        if 'fingerprints' in df_plot.columns:
            df_plot = df_plot.drop(columns=['fingerprints'])

        df_plot['x'] = x
        df_plot['y'] = y

        # Generate molecule images
        print("(generating images...", end=' ')
        df_plot['mol_image'] = df_plot['SMILES'].apply(mol_to_base64)
        print("OK)", end=' ')

        # Create ColumnDataSource
        source = ColumnDataSource(df_plot)

        # Create figure
        p = figure(
            width=900,
            height=700,
            title=f"t-SNE - HIGHDIV - {dataset_name} - {descriptor} ({len(df_plot)} molecules, perplexity={perplexity})",
            tools="pan,wheel_zoom,box_zoom,reset,save",
            toolbar_location="right"
        )

        # Add points
        circles = p.circle(
            'x', 'y',
            size=5,
            alpha=0.6,
            color='steelblue',
            source=source
        )

        # Configure HoverTool with molecule image
        hover = HoverTool(
            tooltips="""
            <div style="width:200px;">
                <div>
                    <img src="@mol_image" style="width:150px; height:150px; border:1px solid #ddd; padding:5px;">
                </div>
                <div style="margin-top:5px;">
                    <span style="font-weight:bold;">Dataset:</span> @Dataset<br>
                    <span style="font-weight:bold;">SMILES:</span> @SMILES<br>
                    <span style="font-weight:bold;">MW:</span> @MW<br>
                    <span style="font-weight:bold;">LogP:</span> @LogP
                </div>
            </div>
            """,
            renderers=[circles]
        )
        p.add_tools(hover)

        # Configure axis labels
        p.xaxis.axis_label = "Dimension 1"
        p.yaxis.axis_label = "Dimension 2"
        p.title.text_font_size = "14pt"

        # Save as HTML
        filename = f"HIGHDIV_{descriptor}_{dataset_name}.html"
        filepath = os.path.join(output_dir, filename)
        output_file(filepath)
        save(p)

        print(f"✓ Saved")

    except Exception as e:
        print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Interactive plots saved in: {output_dir}")
print('='*60)

GENERATING INTERACTIVE PLOTS (HTML)
Plotting: HIGHDIV_MACCS_CONCAT... (generating images... OK) ✓ Saved
Plotting: HIGHDIV_ECFP4_CONCAT... (generating images... 

OK) ✓ Saved
Plotting: HIGHDIV_MACCS_CRAFT... (generating images... 

OK) ✓ Saved
Plotting: HIGHDIV_ECFP4_CRAFT... (generating images... 

OK) ✓ Saved
Plotting: HIGHDIV_MACCS_ENAMINE... (generating images... 

OK) ✓ Saved
Plotting: HIGHDIV_ECFP4_ENAMINE... (generating images... 

OK) ✓ Saved
Plotting: HIGHDIV_MACCS_LANAPDB... (generating images... 

OK) ✓ Saved
Plotting: HIGHDIV_ECFP4_LANAPDB... (generating images... 

OK) ✓ Saved
Plotting: HIGHDIV_MACCS_MAYBRIDGE... (generating images... 

OK) ✓ Saved
Plotting: HIGHDIV_ECFP4_MAYBRIDGE... (generating images... 

OK) ✓ Saved

Interactive plots saved in: /Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/Analysis/Results/Chemical_Spaces/TSNE


## Combined Visualization - All Sub-datasets

In this section, we will create a visualization that combines all sub-datasets (CONCAT, CRAFT, ENAMINE, LANAPDB, MAYBRIDGE) for HIGHDIV, removing duplicates by InChI and maintaining the source sub-dataset information.

In [7]:
# Prepare combined data removing duplicates by InChI
def prepare_combined_data(df_full):
    """
    Concatenates all sub-datasets and removes duplicates by InChI,
    keeping source sub-dataset information
    """
    df_combined = df_full.copy()

    # Remove molecules without InChI
    df_combined = df_combined.dropna(subset=['InChI'])
    print(f"  Molecules with valid InChI: {len(df_combined)}")

    # Remove duplicates by InChI, keeping the first occurrence
    initial_count = len(df_combined)
    df_combined = df_combined.drop_duplicates(subset=['InChI'], keep='first')
    duplicates_removed = initial_count - len(df_combined)
    print(f"  Duplicates removed: {duplicates_removed}")
    print(f"  Unique molecules: {len(df_combined)}")

    return df_combined

print("="*60)
print("PREPARING COMBINED DATA")
print("="*60)

print("\nHIGHDIV:")
df_highdiv_combined = prepare_combined_data(df_highdiv)

print(f"\n{'='*60}")
print("Preparation complete!")
print('='*60)

PREPARING COMBINED DATA

HIGHDIV:
  Molecules with valid InChI: 2712
  Duplicates removed: 1
  Unique molecules: 2711

Preparation complete!


In [8]:
# Calculate t-SNEs for the combined dataset
print("="*60)
print("CALCULATING COMBINED t-SNEs")
print("="*60)

tsne_combined_results = {}

print(f"\nProcessing: HIGHDIV_COMBINED ({len(df_highdiv_combined)} molecules)")
print('='*60)

for descriptor in fingerprint_types:
    print(f"  Calculating t-SNE with {descriptor}...", end=' ')

    try:
        # Create t-SNE with appropriate perplexity
        n_samples = len(df_highdiv_combined)
        perp = min(50, max(5, n_samples // 20))
        layout, df_processed = create_tsne_data(df_highdiv_combined, fingerprint_type=descriptor.lower(), perplexity=perp)

        # Store results
        key = f"HIGHDIV_COMBINED_{descriptor}"
        tsne_combined_results[key] = {
            'layout': layout,
            'df': df_processed,
            'descriptor': descriptor,
            'dataset': 'COMBINED',
            'perplexity': perp
        }

        print(f"✓ Complete ({len(df_processed)} molecules)")

    except Exception as e:
        print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Calculation complete! {len(tsne_combined_results)} combined t-SNEs calculated.")
print('='*60)

CALCULATING COMBINED t-SNEs

Processing: HIGHDIV_COMBINED (2711 molecules)
  Calculating t-SNE with MACCS... 

/opt/anaconda3/envs/ia/lib/python3.13/site-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


[t-SNE] Computing 151 nearest neighbors...
[t-SNE] Indexed 2711 samples in 0.000s...
[t-SNE] Computed neighbors for 2711 samples in 0.410s...
[t-SNE] Computed conditional probabilities for sample 1000 / 2711
[t-SNE] Computed conditional probabilities for sample 2000 / 2711
[t-SNE] Computed conditional probabilities for sample 2711 / 2711
[t-SNE] Mean sigma: 1.767797
[t-SNE] KL divergence after 250 iterations with early exaggeration: 73.925392
[t-SNE] KL divergence after 1000 iterations: 1.276180
✓ Complete (2711 molecules)
  Calculating t-SNE with ECFP4... 

[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerator
[16:18:07] DEPRECATION WARNING: please use MorganGenerat

[t-SNE] Computing 151 nearest neighbors...
[t-SNE] Indexed 2711 samples in 0.001s...
[t-SNE] Computed neighbors for 2711 samples in 0.613s...
[t-SNE] Computed conditional probabilities for sample 1000 / 2711
[t-SNE] Computed conditional probabilities for sample 2000 / 2711
[t-SNE] Computed conditional probabilities for sample 2711 / 2711
[t-SNE] Mean sigma: 2.213292
[t-SNE] KL divergence after 250 iterations with early exaggeration: 73.430717
[t-SNE] KL divergence after 1000 iterations: 1.211890
✓ Complete (2711 molecules)

Calculation complete! 2 combined t-SNEs calculated.


In [9]:
# Generate interactive HTML plots for combined dataset with legend by sub-dataset
print("="*60)
print("GENERATING COMBINED INTERACTIVE PLOTS (HTML)")
print("="*60)

from bokeh.models import Legend, LegendItem

# Bokeh colors for each sub-dataset
dataset_colors_bokeh = {
    'CONCAT':    '#e74c3c',  # red
    'CRAFT':     '#3498db',  # blue
    'ENAMINE':   '#9b59b6',  # purple
    'LANAPDB':   '#2ecc71',  # green
    'MAYBRIDGE': '#f39c12'   # orange
}

dataset_names = ['CONCAT', 'CRAFT', 'ENAMINE', 'LANAPDB', 'MAYBRIDGE']

for key, data in tsne_combined_results.items():
    layout = data['layout']
    df_processed = data['df']
    descriptor = data['descriptor']
    perplexity = data.get('perplexity', 'N/A')

    print(f"Plotting: {key}...", end=' ')

    try:
        # Extract coordinates from t-SNE layout (n_samples, 2)
        x = layout[:, 0]
        y = layout[:, 1]

        # Prepare data for Bokeh
        df_plot = df_processed.copy().reset_index(drop=True)

        # Remove fingerprints column which is not serializable
        if 'fingerprints' in df_plot.columns:
            df_plot = df_plot.drop(columns=['fingerprints'])

        df_plot['x'] = x
        df_plot['y'] = y

        # Generate molecule images
        print("(generating images...", end=' ')
        df_plot['mol_image'] = df_plot['SMILES'].apply(mol_to_base64)
        print("OK)", end=' ')

        # Create figure
        p = figure(
            width=1000,
            height=800,
            title=f"Combined t-SNE - HIGHDIV - {descriptor} ({len(df_plot)} unique molecules by InChI, perplexity={perplexity})",
            tools="pan,wheel_zoom,box_zoom,reset,save",
            toolbar_location="right"
        )

        # Add points by sub-dataset with different colors
        for dataset_name in dataset_names:
            # Filter data for this sub-dataset
            df_subset = df_plot[df_plot['Dataset'].str.contains(dataset_name, case=False, na=False)]

            if len(df_subset) > 0:
                # Create ColumnDataSource for this subset
                source = ColumnDataSource(df_subset)

                # Add circles
                circles = p.circle(
                    'x', 'y',
                    size=7,
                    alpha=0.7,
                    color=dataset_colors_bokeh[dataset_name],
                    source=source,
                    legend_label=f'{dataset_name} (n={len(df_subset)})'
                )

                # Configure HoverTool for this renderer
                hover = HoverTool(
                    tooltips="""
                    <div style="width:200px;">
                        <div>
                            <img src="@mol_image" style="width:150px; height:150px; border:1px solid #ddd; padding:5px;">
                        </div>
                        <div style="margin-top:5px;">
                            <span style="font-weight:bold;">Dataset:</span> @Dataset<br>
                            <span style="font-weight:bold;">SMILES:</span> @SMILES<br>
                            <span style="font-weight:bold;">MW:</span> @MW<br>
                            <span style="font-weight:bold;">LogP:</span> @LogP
                        </div>
                    </div>
                    """,
                    renderers=[circles]
                )
                p.add_tools(hover)

        # Configure axis labels
        p.xaxis.axis_label = "Dimension 1"
        p.yaxis.axis_label = "Dimension 2"
        p.title.text_font_size = "14pt"

        # Configure legend
        p.legend.location = "top_right"
        p.legend.click_policy = "hide"  # Allows hiding when clicking
        p.legend.background_fill_alpha = 0.8

        # Save as HTML
        filename = f"HIGHDIV_COMBINED_{descriptor}.html"
        filepath = os.path.join(output_dir, filename)
        output_file(filepath)
        save(p)

        print(f"✓ Saved")

    except Exception as e:
        print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"Combined interactive plots saved in: {output_dir}")
print('='*60)

GENERATING COMBINED INTERACTIVE PLOTS (HTML)
Plotting: HIGHDIV_COMBINED_MACCS... (generating images... OK) ✓ Saved
Plotting: HIGHDIV_COMBINED_ECFP4... (generating images... 

OK) ✓ Saved

Combined interactive plots saved in: /Users/francisco/Documents/Scripts/DeNovo_PfHDAC1/Analysis/Results/Chemical_Spaces/TSNE
